[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/geraldmc/irilab2026/blob/main/notebooks/r2/r2-q3/01_baseline_and_kitchen_sink.ipynb)

# R2-Q3 Week 2 — Train the baseline and kitchen-sink classifiers

### This notebook produces the two baseline classifiers and their evaluation metrics, the two reference points the targeted augmentation gets compared against next week.

This notebook trains two PV classifiers under the same scaffold: one with no augmentation (the floor) and one with the kitchen-sink list you precommitted in Week 1 (the strong baseline you have to beat). Both get evaluated on PV-internal and on PD. The PV→PD gap you measure for each is the reference number the targeted augmentation gets compared against next week — and the kitchen-sink number is the one to watch, because per the page's Prediction, kitchen-sink may incidentally reduce the gap on its own.

By the end of this notebook you will have:

- Two trained classifiers saved as `baseline_classifier.pkl` and `kitchen_sink_classifier.pkl`, ready for the Week 4 comparison.
- PV-internal and PD evaluation metrics saved as `baseline_metrics.parquet` and `kitchen_sink_metrics.parquet`, each with per-class accuracy and the PV→PD gap.
- Submitted EQ#2.

In [ ]:
# Cell 0 — probe
try:
    import irilab2026 as iri
    print(f"irilab2026 {iri.__version__} already installed.")
    print("If you want to pull the latest changes, run Cell 1 below.")
except ModuleNotFoundError:
    print("irilab2026 not installed — run Cell 1 below.")

In [ ]:
# Cell 1 — install
# First run in a fresh Colab session: run this cell.
# Later runs in the same session: skip this cell.
!pip install git+https://github.com/geraldmc/irilab2026.git@main

**A note about a possible restart prompt.** If Colab shows a "RESTART SESSION" banner after the install: click it, wait for the kernel to reconnect, then continue from the next cell. The install does not need to be run again. If no banner appears, ignore this and move on.

In [ ]:
# Cell 2 — mount and setup
# Mount Drive, set up irilab2026, point to the R2-Q3 output directory.
import irilab2026 as iri
iri.setup()

OUTPUT_DIR = iri.output_dir("r2_q3")
print(f"Output directory: {OUTPUT_DIR}")

### 1) Defensive load of `precommit.json` (so the experimental design and kitchen-sink list drive everything downstream)

This is Week 2. Last week's notebook ended by writing `precommit.json` — your five locked decisions about how R2-Q3's experiment runs. This notebook reads that file back and trains the first two of the three classifiers the experiment compares: one with **no augmentation** (the floor) and one with the **kitchen-sink** augmentation you committed (the strong baseline the targeted set has to beat next week).

Two of the five precommit fields drive this notebook:

- `experimental_design` — confirms you're running the three-condition factorial comparison these notebooks are built around.
- `kitchen_sink` — the exact augmentation policy this notebook builds and trains with.

The reason to *load* those decisions rather than retype them is the whole point of having precommitted: the augmentation you train with this week should be the one you reasoned about and locked last week, not a fresh choice made mid-experiment. If you find yourself wanting to change a precommit value now, that is a change to your methodology — make it in NB00 and re-run, so the record stays honest.

In [ ]:
# Read back the five decisions you locked in NB00. This notebook consumes
# two of them; the load is defensive so a missing or half-written file
# fails here, with a clear message, rather than partway through training.
import json

precommit_path = OUTPUT_DIR / "precommit.json"
if not precommit_path.exists():
    raise FileNotFoundError(
        f"No precommit.json at {precommit_path}. "
        "Run NB00 (00_orient_and_precommit.ipynb) to completion first — "
        "it writes the Week-1 decisions this notebook reads."
    )

with open(precommit_path) as f:
    precommit = json.load(f)

# NB01 consumes two of the five precommit fields:
#   experimental_design — confirms the three-condition factorial setup
#                         this notebook chain is built around.
#   kitchen_sink        — the augmentation policy Section 2 builds.
for key in ("experimental_design", "kitchen_sink"):
    if precommit.get(key) is None:
        raise KeyError(
            f"precommit.json is missing '{key}'. Re-run NB00 — every field "
            "should be populated before you start Week 2."
        )

design = precommit["experimental_design"]
ks = precommit["kitchen_sink"]

# This chain (NB01-NB03) is scaffolded for the factorial design: three named
# conditions compared head to head. NB01 trains two of them (no-augmentation
# and kitchen-sink); NB02 trains the third (targeted). A hold-one-out design
# would need a different training loop than NB01 provides, so flag it rather
# than silently running the wrong shape.
if design["design"] != "factorial":
    print(
        f"WARNING: experimental_design is {design['design']!r}, not "
        "'factorial'. These notebooks train and compare the three named "
        "factorial conditions; hold-one-out needs a different loop. Check "
        "with your mentor before continuing."
    )

# Pull out the kitchen-sink parameters Section 2 passes to
# iri.randaugment_train_transform(...). Reading them from the file (rather
# than retyping) is what guarantees you train with the policy you locked.
if ks["method"] == "randaugment":
    ks_params = ks["randaugment_params"]
else:
    ks_params = None  # fixed_list path — handled in Section 2.

# Print back what drives the rest of the notebook.
print("Loaded precommit.json")
print(f"  Experimental design: {design['design']} "
      f"({design['n_conditions']} conditions)")
if ks["method"] == "randaugment":
    print(f"  Kitchen-sink:        RandAugment("
          f"num_ops={ks_params['num_ops']}, magnitude={ks_params['magnitude']})")
else:
    n = len(ks["transforms"]) if ks.get("transforms") else 0
    print(f"  Kitchen-sink:        fixed list ({n} transforms)")

Before training, set how this run executes. There is one knob, `DEV_MODE`, and it does two things at once: which size of PlantVillage to load, and how many training epochs to run.

Leave `DEV_MODE = True` while you are getting the notebook to run end to end — it uses a small slice of PlantVillage and a short two-epoch cap, so both training runs finish in a few minutes and you catch mistakes cheaply. Switch to `DEV_MODE = False` for the real run: the full dataset and the full ten-epoch recipe, which takes tens of minutes on a GPU but produces the numbers you will report.

Whatever you choose, both training runs use the same setting. That is deliberate — the only difference allowed between the two classifiers is the augmentation, so dataset size and epoch count are held identical. Training also requires a GPU; the cell checks for one and stops early, with instructions, if it does not find it.

In [ ]:
# Run configuration — set once here; every section below reads from it.
#
# DEV_MODE is the one knob you flip while getting the notebook running:
#   True  -> the "tiny" PV variant (~50 images/class) and a 2-epoch cap.
#            Both training runs finish in a few minutes.
#   False -> the "full" PV variant (~54k images) and the full 10-epoch
#            recipe. The real run: tens of minutes on a GPU.
#
# Both training runs use the SAME variant and epoch cap. That is on purpose:
# the only thing allowed to differ between the no-augmentation and
# kitchen-sink classifiers is the augmentation. Holding the data and the
# epoch count identical is what makes the gap comparison fair.
DEV_MODE = True

PV_VARIANT = "tiny" if DEV_MODE else "full"
EPOCH_CAP = 2 if DEV_MODE else None   # None -> train_baseline's full 10 epochs

# One seed for the whole notebook. train_baseline also seeds itself per call
# from its own `seed=` argument; seeding here keeps anything outside that call
# (data loading, evaluation ordering) reproducible too.
iri.seed_all(42)

# Training calls .cuda() directly, so a GPU is not optional in this notebook.
# Fail now, with a clear message, rather than partway through training with a
# cryptic CUDA error.
assert iri.has_gpu(), (
    "No GPU detected. This notebook trains two ResNet-18 classifiers and "
    "needs a GPU. In Colab: Runtime -> Change runtime type -> T4 GPU, then "
    "re-run from the top."
)

print(f"DEV_MODE   = {DEV_MODE}")
print(f"PV_VARIANT = {PV_VARIANT!r}")
print(f"EPOCH_CAP  = {EPOCH_CAP}")
print("Seed set to 42; GPU detected.")

### 2) Build the two data pipelines: no augmentation, and the precommitted kitchen-sink list

You will set the two conditions up as a short list: each entry is a name paired with the train transform that defines it, and Section 3 trains one classifier per entry. Building them as a list — instead of writing the training out twice — is what keeps the two runs identical apart from the transform.

This section builds the first transform and leaves the second for you. The first is the no-augmentation floor, and it pays to be precise about what "no augmentation" means here, because the obvious choice is the wrong one. The default training transform, `imagenet_train_transform()`, already augments — it takes a random crop and randomly flips each image. The transform with genuinely *no* augmentation is the evaluation transform, `imagenet_eval_transform()`: a fixed resize and center-crop, identical every time, no randomness. That deterministic pipeline is the true floor, so it is what you pass as the training transform for the baseline condition.

The cell below loads PlantVillage once — both conditions train on the same images — reads the number of classes straight from the data, and builds that floor transform.

In [ ]:
# Load PlantVillage once. Both conditions train on the same PV data; only the
# transform differs. PV_VARIANT came from the run-configuration cell in
# Section 1 ("tiny" in DEV_MODE, "full" otherwise).
metadata, hf_dataset = iri.load_plantvillage(variant=PV_VARIANT)

# Read the number of classes from the data rather than hardcoding 38. If the
# variant ever changes, this stays correct.
num_classes = metadata["class_idx"].nunique()

# The no-augmentation floor: the deterministic eval transform, used as the
# TRAIN transform here. (The default train transform augments via random crop
# and flip — see the note above — so the eval transform is the true floor.)
baseline_transform = iri.imagenet_eval_transform()

n_train = (metadata["split"] == "train").sum()
print(f"PlantVillage ({PV_VARIANT}): {len(metadata)} images total, "
      f"{n_train} in the train split, {num_classes} classes.")
print("Built baseline_transform (no augmentation: deterministic eval pipeline).")

The second condition is the **kitchen-sink** baseline: a broad, general-purpose augmentation applied without reference to any particular failure mode. The point of a kitchen-sink is that you do not hand-pick the transformations — you let a standard policy throw a bit of everything at the image, and see how far that alone closes the gap. The targeted set you build next week has to beat this.

The standard policy is **RandAugment** (Cubuk et al., 2020). On each image it applies a few randomly chosen operations — rotate, shear, color shift, contrast, and so on — drawn from a fixed menu, each at a shared strength. You do not choose the operations; that is the whole idea. It exposes two knobs:

- `num_ops` — how many operations are applied per image.
- `magnitude` — how strong each one is, on a 0–30 scale.

The library wraps this for you as `iri.randaugment_train_transform(num_ops=..., magnitude=...)`, which slots RandAugment into the pipeline at the right place (it has to run on the image before it becomes a tensor) so you do not have to reason about ordering. The two values come from your Week-1 precommit — you loaded them into `ks_params` back in Section 1.

### Practice 2.1 — Build the kitchen-sink transform

Build the kitchen-sink training transform and assign it to a variable named `kitchen_sink_transform`. Use the library helper, and use the values you committed in Week 1 — they are sitting in `ks_params` from Section 1, so pass those rather than retyping the numbers. The next cell expects `kitchen_sink_transform` to exist; if you skip this, it will stop with a `NameError`.

*Hint: `iri.randaugment_train_transform(...)` takes `num_ops` and `magnitude` as keyword arguments. `ks_params` is a dictionary with exactly those two keys.*

In [ ]:
### Practice 2.1 — your code here --------------------------------------------------
#
#
#
#
#
#
# -----------------------------------------------------------------------------------

In [ ]:
# Assemble the two conditions Section 3 will loop over. Each is a name paired
# with the train transform that defines it. Everything else about training is
# held identical across the two — that is what makes the comparison fair.
conditions = [
    {"name": "baseline",     "train_transform": baseline_transform},
    {"name": "kitchen_sink", "train_transform": kitchen_sink_transform},
]

print(f"{len(conditions)} conditions ready to train:")
for cond in conditions:
    t = cond["train_transform"]
    # Name the transform in a way the student can recognize: the floor is the
    # eval transform; the kitchen-sink is a RandAugment Compose.
    label = ("no augmentation (eval transform)"
             if t is baseline_transform
             else f"RandAugment(num_ops={ks_params['num_ops']}, "
                  f"magnitude={ks_params['magnitude']})")
    print(f"  - {cond['name']}: {label}")

### 3) Train both classifiers

With both conditions defined, training is the same single step run once per condition. The library function `iri.train_baseline` carries the whole training recipe — it builds a ResNet-18 with ImageNet-pretrained weights and a fresh classifier head, carves a 10% validation slice out of the training data, runs the optimizer for the set number of epochs, and keeps the version of the model that scored best on that validation slice. You hand it the data, the number of classes, and the one thing that changes between conditions — the train transform — and it hands back two things: the trained model's weights and a short history of how training went.

Two details to hold onto. First, you pass the *full* metadata frame; `train_baseline` selects the training split and carves its own validation slice internally, so you do not pre-split it yourself. Second, both conditions run with the same seed (42). That is deliberate: the same seed gives both classifiers the same starting weights and the same train/validation split, so the only thing that differs between them is the augmentation — which is exactly the comparison you want.

How long this takes depends on the `DEV_MODE` setting from Section 1: a few minutes per condition on the tiny variant, roughly 20–25 minutes per condition on the full one (so budget the better part of an hour for both full runs on a T4 GPU). The per-epoch lines printed below show training loss, validation loss, and validation accuracy as each run progresses.

In [ ]:
# Train one classifier per condition. The only argument that changes between
# iterations is train_transform; everything else is held identical, which is
# what makes the two classifiers comparable.
results = {}
for cond in conditions:
    name = cond["name"]
    print(f"=== Training: {name} ===")
    state_dict, history = iri.train_baseline(
        metadata,
        hf_dataset,
        dataset_class=iri.PlantVillageDataset,
        num_classes=num_classes,
        train_transform=cond["train_transform"],
        seed=42,              # same seed for both -> same init and val split
        epoch_cap=EPOCH_CAP,  # 2 in DEV_MODE, None (full 10 epochs) otherwise
    )
    results[name] = {"state_dict": state_dict, "history": history}

In [ ]:
# Best validation accuracy reached by each condition, on PV's held-out
# validation slice. This is a quick training-health check, not the headline
# result — the comparison that matters is the PV->PD gap, computed next.
print("Training complete. Best PV-validation accuracy by condition:")
for name, r in results.items():
    h = r["history"]
    print(f"  {name:12s} {h['best_val_acc']:.4f}  (epoch {h['best_val_epoch'] + 1})")

A note on what you are looking at and what comes next. The validation accuracy above is measured on a slice of PlantVillage held out from training — so it tells you each classifier learned PlantVillage, not whether it transfers to field photographs. The number that actually answers R2-Q3 is the drop from PlantVillage to PlantDoc, and you compute that over the next two sections: Section 4 measures each classifier on held-out PlantVillage, Section 5 measures the same classifiers on PlantDoc and takes the difference.

One practical point for those sections: `train_baseline` handed back each model's *weights* (a state dictionary), not a ready-to-run model. Carrying a trained model around as its weights is the normal pattern — it is small, portable, and easy to save. In Section 4 you load those weights into a fresh ResNet-18 to turn them back into a model you can evaluate.

### 4) Evaluate both on PV-internal: per-class accuracy on the held-out PV split

### 5) Evaluate both on PD; compute the PV→PD gap for each

### 6) Closeout: save `baseline_classifier.pkl`, `kitchen_sink_classifier.pkl`, `baseline_metrics.parquet`, and `kitchen_sink_metrics.parquet`; submit EQ#2